# Exploratory Data Analysis: Rossmann Store Sales

This notebook performs initial data exploration and validation for the Rossmann Store Sales dataset. The analysis focuses on understanding feature distributions, temporal seasonality, and the impact of promotional activities to inform baseline model selection.

### Objectives:
- Validate raw data against the defined schema.
- Analyze the distribution of the target variable (Sales).
- Evaluate seasonal trends by Day of Week.
- Assess the correlation between promotions and sales volume.


In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import yaml

# Add project root to path for local imports
sys.path.append(str(Path.cwd().parent))
from src.data_validation import validate_data

# Visual configuration
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

# Load configuration parameters
with open("../configs/params.yaml", "r") as f:
    params = yaml.safe_load(f)

print("Environment setup complete.")

In [ ]:
# Load raw training and store metadata
train_df = pd.read_csv(f"../{params['data']['raw_train']}", low_memory=False)
store_df = pd.read_csv(f"../{params['data']['raw_store']}")

# Merge datasets on Store ID
df = pd.merge(train_df, store_df, on="Store", how="left")
print(f"Merged Dataset Dimensions: {df.shape}")
df.head()

### Data Validation
Verifying the merged dataset against the Pandera schema to ensure data integrity and detect any anomalies or poisoned records.


In [ ]:
try:
    df_validated = validate_data(df)
    print("Dataset validation passed successfully.")
except Exception as error:
    print(f"Dataset validation failed: {error}")

### Sales Distribution
Analysis of the target variable `Sales` to identify skewness and the impact of store closures (Open=0).


In [ ]:
plt.figure(figsize=(10, 5))
sns.histplot(df["Sales"], bins=50, kde=True)
plt.title("Distribution of Daily Sales Portfolio")
plt.xlabel("Sales Volume")
plt.ylabel("Frequency")
plt.show()

# Quantify zero-sales impact
closed_sales_avg = df[df["Open"] == 0]["Sales"].mean()
print(f"Mean Sales (Closed Stores): {closed_sales_avg}")
print(f"Total entries with zero sales: {(df['Sales'] == 0).sum()}")

### Temporal Analysis: Day of Week
Examination of sales fluctuations across the week and the relative lift generated by promotional activities.


In [ ]:
sns.barplot(data=df, x="DayOfWeek", y="Sales", hue="Promo")
plt.title("Sales Performance by Day of Week and Promotion Status")
plt.xlabel("Day of Week (1=Mon, 7=Sun)")
plt.ylabel("Average Sales")
plt.show()

### Analysis Summary & Modeling Implications

- **Zero-Sales Variance**: The histogram reveals a massive zero-sales peak, primarily attributed to days when stores are closed (`Open=0`). Including these rows in regression training would introduce significant noise, as the model would struggle to learn the variance of "closed" stores which are structurally zero.
- **Promo Efficiency**: The categorical analysis confirms that promotions are highly effective but predominantly scheduled for weekdays (1-5). Monday (Day 1) shows the strongest response to promotions, while Friday (Day 5) exhibits the highest baseline sales among weekdays without promotions.
- **Sunday Exception**: Sunday (Day 7) sales are near-zero on average, and promotions are never active on this day in the training set. This structural behavior supports a two-stage modeling approach: filtering for open stores during training and post-processing zero-sales for Sundays/closures.
- **Skewness & Scaling**: The target variable `Sales` exhibits a right-skewed distribution with a long tail reaching 40,000+. We may need to consider log-transformation or robust scaling for better convergence in linear models.
- **Reproducibility**: Pandera validation confirmed that `Sales`, `Promo`, and `DayOfWeek` columns are within acceptable bounds, ensuring no immediate data quality issues prevent training.
